<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">

# yoctoGPT — Mixed Corpus Training

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

This notebook demonstrates **combined-corpus training** with `yoctoGPT`:
1. Train one BPE tokenizer on a combined corpus (general English + finance texts).
2. **Large training run** on the full combined corpus.
3. Sample with both a philosophy prompt and a finance prompt.
4. **Resume training** from the best checkpoint for additional refinement.
5. Sample again with the same prompts and compare.

## How to Use This Notebook

- **Goal**: Train a single model on a combined general + finance corpus, then resume for further refinement.
- **Requirements**: Text files in `data/` (general English) and `finance/` (domain-specific).
- **Hardware**: Designed for large GPUs (G4, A100, H100). GPU is auto-detected and training parameters are scaled accordingly via the `transfer` profile in `gpu_configs.yml`.
- **Persistence**: Mounts Google Drive for persistent checkpoint storage.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the full GPU comparison table.

### Roadmap

1. **Setup**: Install dependencies, clone repo, mount Google Drive.
2. **Corpus Preparation**: Clean general and finance texts, train a combined BPE tokenizer.
3. **Large Training Run**: Train on the full combined corpus.
4. **Sampling (After Large Run)**: Generate text with philosophy and finance prompts.
5. **Resume Training**: Resume from the best checkpoint with a lower learning rate.
6. **Sampling (After Resume)**: Same prompts, compare text quality evolution.

In [ ]:
#@title Mount Google Drive for checkpoint storage (repo stays local)
AUTO_DISCONNECT = True  #@param {type: 'boolean'}

from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

CKPT_DIR = Path('/content/drive/MyDrive/yocto/checkpoints/mixed_corpus')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoints dir:', CKPT_DIR)

In [ ]:
#@title Setup: Install Dependencies and Clone Repository
!nvidia-smi || true
!pip -q install tokenizers tqdm textstat pyyaml

import os, pathlib, subprocess

repo_root = pathlib.Path('/content/yoctoGPT')
if repo_root.exists():
    print('Repo exists, pulling latest...')
    subprocess.run(['git', 'pull'], cwd=repo_root, check=False)
else:
    subprocess.run(['git', 'clone', 'https://github.com/yhilpisch/yoctoGPT.git', str(repo_root)], check=False)
os.chdir(repo_root)

if os.path.exists('requirements.txt'):
    !pip -q install -r requirements.txt || true

# Verify both corpora exist
for name, d in [('general', 'data'), ('finance', 'finance')]:
    p = pathlib.Path(d)
    if not p.exists() or not list(p.glob('*.txt')):
        raise FileNotFoundError(f'No .txt files found in {d}/. Add {name} text files before running.')
    txts = sorted(p.glob('*.txt'))
    print(f'Found {len(txts)} {name} text file(s) in {d}/')
    for fp in txts[:5]:
        print(' -', fp.name)
    if len(txts) > 5:
        print(' - ...')

In [ ]:
#@title Detect GPU and load training profile
import subprocess, yaml

_out = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip()

if "T4" in _out:
    GPU_KEY = "t4"
elif "L4" in _out:
    GPU_KEY = "l4"
elif "A100" in _out:
    GPU_KEY = "a100"
elif "RTX PRO 6000" in _out:
    GPU_KEY = "g4"
elif "H100" in _out:
    GPU_KEY = "h100"
else:
    GPU_KEY = "t4"

GPU_CFG = yaml.safe_load(open("notebooks/gpu_configs.yml"))[GPU_KEY]
AMP_DTYPE = GPU_CFG["amp_dtype"]
print(f"GPU: {_out} -> profile: {GPU_KEY}, amp_dtype: {AMP_DTYPE}")

### Corpus Preparation

We clean both corpora and train **one BPE tokenizer** on the combined text.
The tokenizer covers subwords from both general English and finance domains.

The `prepare_tokenizer` script with `--all_txt_in_dir` encodes the full
combined corpus into a single dataset at `data/token_transfer/`:
- `train.bin` and `val.bin` — the combined token arrays for training.
- `tokenizer.json` — the shared BPE tokenizer.

In [ ]:
#@title Clean both corpora and train combined tokenizer
# Clean general texts
!python -m scripts.prepare_book_text \
--in_dir data \
--out_dir data_clean \
--glob '*.txt' \
--lowercase

# Clean finance texts (drop formulas)
!python -m scripts.prepare_book_text \
--in_dir finance \
--out_dir finance_clean \
--glob '*.txt' \
--lowercase \
--drop_formulas

# Combine into one directory for tokenizer training
import shutil, pathlib
combined = pathlib.Path('combined_clean')
if combined.exists():
    shutil.rmtree(combined)
combined.mkdir()
for src_dir in [pathlib.Path('data_clean'), pathlib.Path('finance_clean')]:
    for fp in sorted(src_dir.glob('*.txt')):
        shutil.copy2(fp, combined / fp.name)

# Train tokenizer on combined corpus (also encodes train.bin / val.bin)
!python -m scripts.prepare_tokenizer \
--all_txt_in_dir \
--text_dir combined_clean \
--out_dir data/token_transfer \
--vocab_size 4000 \
--backend bpe \
--random_split \
--split_level chunk \
--split_seed 1337 \
--add_bos_eos

In [ ]:
#@title Check corpus size and load transfer profile
from pathlib import Path
import numpy as np

train_path = Path('data/token_transfer/train.bin')
val_path = Path('data/token_transfer/val.bin')
train_tokens = int(np.fromfile(train_path, dtype=np.int32).shape[0])
val_tokens = int(np.fromfile(val_path, dtype=np.int32).shape[0])
print(f'Combined corpus: {train_tokens:,} train, {val_tokens:,} val tokens')

P = GPU_CFG["transfer"]
n_layer, n_head = P["n_layer"], P["n_head"]
n_embd = P["n_embd"]
block_size, batch_size = P["block_size"], P["batch_size"]
print(f"\ntransfer: n_layer={n_layer}, n_head={n_head}, n_embd={n_embd}, "
      f"block_size={block_size}, batch_size={batch_size}")

### Large Training Run

Train on the full combined corpus (general English + finance).
The model learns both broad language patterns and domain-specific
finance vocabulary in a single pass.

In [ ]:
#@title Large training run on combined corpus
LARGE_DIR = CKPT_DIR / 'large'
LARGE_DIR.mkdir(parents=True, exist_ok=True)

!python -m yoctoGPT.train \
--mode token \
--data_dir data/token_transfer \
--tokenizer_path data/token_transfer/tokenizer.json \
--ckpt_dir {LARGE_DIR} \
--model_type gpt_fast \
--device cuda \
--n_layer {n_layer} --n_head {n_head} --n_embd {n_embd} \
--block_size {block_size} --batch_size {batch_size} \
--dropout 0.12 --weight_decay 0.08 \
--tie_weights --label_smoothing 0.05 \
--amp --amp_dtype {AMP_DTYPE} \
--auto_microbatch \
--eval_interval 250 --eval_iters 50 \
--cosine_lr --warmup_iters 300 \
--min_lr 1e-5 --lr 1.8e-4 \
--max_iters 2000 \
--ema --ema_decay 0.999

### Sampling (After Large Run)

We sample with two prompts — one philosophical, one financial — to see how the combined model handles each domain.

In [ ]:
#@title Sample — Philosophy prompt (after large run)
PROMPT_PHIL = "In the beginning, philosophy sought to"
output_phil_large = !python -m yoctoGPT.sampler \
--mode token \
--ckpt {LARGE_DIR}/best.pt \
--tokenizer_path data/token_transfer/tokenizer.json \
--prompt "{PROMPT_PHIL}" \
--max_new_tokens 120 \
--temperature 0.85 --top_k 40 --top_p 0.90

text_phil_large = '\n'.join(output_phil_large)
print(text_phil_large)

In [ ]:
#@title Sample — Finance prompt (after large run)
PROMPT_FIN = "in financial theory, the principle of no arbitrage implies"
output_fin_large = !python -m yoctoGPT.sampler \
--mode token \
--ckpt {LARGE_DIR}/best.pt \
--tokenizer_path data/token_transfer/tokenizer.json \
--prompt "{PROMPT_FIN}" \
--max_new_tokens 120 \
--temperature 0.85 --top_k 40 --top_p 0.90

text_fin_large = '\n'.join(output_fin_large)
print(text_fin_large)

### Text Quality Assessment (After Large Run)

Score both samples against the combined reference corpus.

In [ ]:
#@title Score Generated Text Quality (after large run)
from yoctoGPT.text_metrics import score_text, print_scorecard

reference_corpus = '\n\n'.join(
    p.read_text(encoding='utf-8')
    for p in sorted(pathlib.Path('combined_clean').glob('*.txt'))
)

print('=== Philosophy prompt (after large run) ===')
card_phil_large = score_text(text_phil_large, reference_text=reference_corpus)
print_scorecard(card_phil_large)

print('\n=== Finance prompt (after large run) ===')
card_fin_large = score_text(text_fin_large, reference_text=reference_corpus)
print_scorecard(card_fin_large)

### Resume Training

Resume from the best checkpoint (`--resume`) and continue training on the
same combined corpus with a **lower learning rate**.

This refines the model further: the cosine schedule restarts with a
smaller peak LR, letting the optimiser settle into a sharper minimum
without disturbing what the large run already learned.

In [ ]:
#@title Resume training from best checkpoint
RESUME_DIR = CKPT_DIR / 'resume'
RESUME_DIR.mkdir(parents=True, exist_ok=True)

best = LARGE_DIR / 'best.pt'
if not best.exists():
    print(f'No checkpoint found at {best}; run the large training cell first.')
else:
    !python -m yoctoGPT.train \
    --mode token \
    --data_dir data/token_transfer \
    --tokenizer_path data/token_transfer/tokenizer.json \
    --ckpt_dir {RESUME_DIR} \
    --resume {best} \
    --model_type gpt_fast \
    --device cuda \
    --n_layer {n_layer} --n_head {n_head} --n_embd {n_embd} \
    --block_size {block_size} --batch_size {batch_size} \
    --dropout 0.12 --weight_decay 0.08 \
    --tie_weights --label_smoothing 0.05 \
    --amp --amp_dtype {AMP_DTYPE} \
    --auto_microbatch \
    --eval_interval 250 --eval_iters 60 \
    --cosine_lr --warmup_iters 100 \
    --min_lr 1e-5 --lr 8e-5 \
    --max_iters 1200 \
    --ema --ema_decay 0.999

### Sampling (After Resume)

Same two prompts as before. Compare the output with the large-run samples to see how resume training refined the model.

In [ ]:
#@title Sample — Philosophy prompt (after resume)
output_phil_resume = !python -m yoctoGPT.sampler \
--mode token \
--ckpt {RESUME_DIR}/best.pt \
--tokenizer_path data/token_transfer/tokenizer.json \
--prompt "{PROMPT_PHIL}" \
--max_new_tokens 120 \
--temperature 0.85 --top_k 40 --top_p 0.90

text_phil_resume = '\n'.join(output_phil_resume)
print(text_phil_resume)

In [ ]:
#@title Sample — Finance prompt (after resume)
output_fin_resume = !python -m yoctoGPT.sampler \
--mode token \
--ckpt {RESUME_DIR}/best.pt \
--tokenizer_path data/token_transfer/tokenizer.json \
--prompt "{PROMPT_FIN}" \
--max_new_tokens 120 \
--temperature 0.85 --top_k 40 --top_p 0.90

text_fin_resume = '\n'.join(output_fin_resume)
print(text_fin_resume)

### Text Quality Assessment (After Resume)

Score both samples and compare with the large-run scores above.
Both prompts should improve or hold steady, showing that resume
training refines the model without losing either domain.

In [ ]:
#@title Score Generated Text Quality (after resume)
print('=== Philosophy prompt (after resume) ===')
card_phil_resume = score_text(text_phil_resume, reference_text=reference_corpus)
print_scorecard(card_phil_resume)

print('\n=== Finance prompt (after resume) ===')
card_fin_resume = score_text(text_fin_resume, reference_text=reference_corpus)
print_scorecard(card_fin_resume)

print('\n=== Comparison (large run -> resume) ===')
for label, large, resume in [('Philosophy', card_phil_large, card_phil_resume),
                              ('Finance', card_fin_large, card_fin_resume)]:
    kl_l = large.get('Char-KL', float('nan'))
    kl_r = resume.get('Char-KL', float('nan'))
    d1_l = large.get('Distinct-1', float('nan'))
    d1_r = resume.get('Distinct-1', float('nan'))
    print(f'{label:12s}  Char-KL: {kl_l:.4f} -> {kl_r:.4f}  '
          f'Dist-1: {d1_l:.4f} -> {d1_r:.4f}')

### Exercises

1. **Resume Duration**: Reduce `--max_iters` to 600 or increase to 2000. How does the text quality change? Does longer resume help or start to overfit?
2. **Learning Rate**: Try `--lr 1e-4` (higher) or `--lr 4e-5` (lower) for the resume run. Higher LR adapts faster but may destabilise; lower LR is safer but slower.
3. **Vocab Size**: Re-run with `--vocab_size 4000` or `12000` during tokenizer training. How does vocab size affect both domains simultaneously?
4. **Model Size**: On a G4 or A100, try a larger model by increasing `n_layer` and `n_embd` in the `transfer` profile. Does the bigger model overfit the 3M-token combined corpus?

In [ ]:
#@title Auto-disconnect runtime (frees Colab compute when finished)
if AUTO_DISCONNECT:
    print('All cells complete. Disconnecting runtime...')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_DISCONNECT is False. Runtime stays active.')

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">